# AMF Fault Detector: LSTM-Autoencoder (Section VII.D, Table 10)

**Self-contained.** This notebook bundles the SCOPE-5G generator from `AMF_Pipeline_Complete.ipynb`, generates the 60-day dataset under seed=42, and trains an LSTM-autoencoder fault detector under two evaluation splits (S1 default, S2 temporal-generalization).

**Purpose.** Reproduces the LSTM-AE rows of Table 10 in Section VII.D of the revised manuscript, plus the temporal-generalization (S2) IF row.

**Architecture.**
- Input: overlapping 96-slot (24-hour) sliding windows over the 102 native TS 28.552 columns
- Encoder: 2 stacked LSTM layers, 128 units each → 64-dim bottleneck
- Decoder: symmetric — 64 → 2×128 LSTM → 102-feature reconstruction
- Loss: MSE; anomaly score: per-row reconstruction RMSE; threshold: 99th percentile of clean-training reconstruction error

**Splits.**
- (S1) **Default clean-baseline:** train days 1–2 and 50–60, test days 3–49
- (S2) **Temporal-generalization:** train days 1–2 only, test days 3–60

**Just press Run All.** No edits required. The numbers you need to paste are printed at the bottom.

**Runtime:** ~5 min on GPU (Colab T4), ~30 min on CPU. To enable GPU: Runtime → Change runtime type → T4 GPU.

## Part 1 — Generator code (bundled from AMF_Pipeline_Complete.ipynb)

The cells below are an exact copy of the generator. Just run them.

In [1]:
import os, io, json, warnings, zipfile, subprocess, glob, math, time, types
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt, matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import (ks_2samp, pearsonr, spearmanr, mannwhitneyu,
                          wasserstein_distance, shapiro, nbinom)
from scipy.signal import periodogram
from scipy.interpolate import interp1d
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Union

warnings.filterwarnings('ignore', category=RuntimeWarning, module='scipy')
warnings.filterwarnings('ignore', category=FutureWarning,  module='sklearn')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn')
matplotlib.rcParams.update({'figure.dpi': 130, 'font.size': 9})
print('All imports OK.')

All imports OK.


In [2]:
# Configuration: EVENT_DATA, SLICE params, seeds, SLICE_CPU_MULT
RANDOM_SEED    = 42
DURATION_HOURS = 1440         # 60 days — released benchmark
# For ablation / rapid iteration use DURATION_HOURS=336 (14 days)
STEP_MIN       = 15           # 15-minute PM granularity (3GPP standard)
AMF_INSTANCES  = 1            # released benchmark (framework supports N≥1)
NUM_UES        = 100_000
VCPUS_PER_AMF  = 8
MEM_MAX_MB     = 8192.0       # 8 GB RAM per AMF VNF instance
HURST_EXPONENT = 0.75         # long-range dependence (self-similar traffic)

OUTPUT_DIR     = "/tmp/amf_dataset"

# ─── 3GPP procedure event catalogue ────────────────────────────────────────
# bh_rate : mean events per UE per busy hour (calibrated from operational AMF
#            measurements and 3GPP TR 23.700-81 reference load models)
# msgs    : N1+N2 messages per procedure (3GPP TS 23.502 call flow message counts)
# nb_k    : Negative-Binomial dispersion parameter k  (larger k → Poisson limit)
# cpu_w   : CPU weight relative to Service Request = 1.0
#           Derived from Table 4 of Chiha et al. (2020): instruction counts
#           measured on a commercial AMF:
#             ServiceRequest  ≈ 3 580 k instr  → reference 1.00
#             ServiceRelease  ≈ 3 200 k instr  → 0.89
#             XnHandover      ≈ 2 140 k instr  → 0.60
#             InitReg         ≈ 4 200 k instr  → 1.17 (auth + key derivation)
# mem_ctx : UE-context memory footprint per procedure (MB per active context)
#           Base: ~5 KB NAS state; +2 KB security context per auth proc.
EVENT_DATA: Dict[str, Dict] = {
    "Initial Registration":      {"bh_rate": 0.625,  "msgs": 12, "nb_k": 60, "cpu_w": 1.17, "mem_ctx": 0.007},
    "Deregistration":            {"bh_rate": 0.208,  "msgs":  4, "nb_k": 50, "cpu_w": 0.45, "mem_ctx": 0.000},
    "Inter-AMF Mobility Reg":    {"bh_rate": 0.101,  "msgs":  8, "nb_k": 40, "cpu_w": 0.89, "mem_ctx": 0.005},
    "Intra-AMF Mobility Reg":    {"bh_rate": 1.575,  "msgs":  4, "nb_k": 70, "cpu_w": 0.60, "mem_ctx": 0.004},
    "Periodic Registration":     {"bh_rate": 0.5103, "msgs":  4, "nb_k": 80, "cpu_w": 0.20, "mem_ctx": 0.001},
    "Service Request":           {"bh_rate": 24.261, "msgs":  4, "nb_k": 50, "cpu_w": 1.00, "mem_ctx": 0.002},
    "PS Paging":                 {"bh_rate": 10.666, "msgs":  2, "nb_k": 45, "cpu_w": 0.15, "mem_ctx": 0.000},
    "N2 Release":                {"bh_rate": 30.000, "msgs":  2, "nb_k": 55, "cpu_w": 0.89, "mem_ctx": 0.000},
    "Inter-AMF N2 Handover":     {"bh_rate": 1.567,  "msgs": 10, "nb_k": 35, "cpu_w": 0.80, "mem_ctx": 0.005},
    "Intra-AMF N2 Handover":     {"bh_rate": 6.267,  "msgs":  6, "nb_k": 40, "cpu_w": 0.70, "mem_ctx": 0.003},
    "Intra-AMF Xn Handover":     {"bh_rate": 14.099, "msgs":  4, "nb_k": 45, "cpu_w": 0.60, "mem_ctx": 0.002},
    "PDU Session Establishment": {"bh_rate": 1.359,  "msgs":  6, "nb_k": 60, "cpu_w": 0.70, "mem_ctx": 0.001},
    "PDU Session Release":       {"bh_rate": 0.507,  "msgs":  4, "nb_k": 55, "cpu_w": 0.40, "mem_ctx": 0.000},
    "PDU Session Modification":  {"bh_rate": 2.063,  "msgs":  4, "nb_k": 60, "cpu_w": 0.55, "mem_ctx": 0.001},
    "VoNR Voice Call":           {"bh_rate": 0.2293, "msgs":  4, "nb_k": 30, "cpu_w": 0.75, "mem_ctx": 0.003},
    "EPS Fallback Voice":        {"bh_rate": 0.6412, "msgs":  4, "nb_k": 35, "cpu_w": 0.60, "mem_ctx": 0.002},
    "SMS":                       {"bh_rate": 0.4279, "msgs":  4, "nb_k": 40, "cpu_w": 0.25, "mem_ctx": 0.001},
    "5GS-to-EPS Mobility":       {"bh_rate": 1.000,  "msgs":  8, "nb_k": 30, "cpu_w": 0.80, "mem_ctx": 0.004},
    "EPS-to-5GS Mobility":       {"bh_rate": 3.000,  "msgs":  8, "nb_k": 35, "cpu_w": 0.80, "mem_ctx": 0.004},
    "5GS-to-EPS HO (N26)":       {"bh_rate": 2.000,  "msgs": 10, "nb_k": 28, "cpu_w": 0.90, "mem_ctx": 0.005},
    "EPS-to-5GS HO (N26)":       {"bh_rate": 0.000,  "msgs": 10, "nb_k": 28, "cpu_w": 0.90, "mem_ctx": 0.005},
}

# ─── Default slice population mix ──────────────────────────────────────────
SERVICE_MIX = {"eMBB": 0.70, "mMTC": 0.20, "URLLC": 0.10}

# ─── Per-slice arrival rate multipliers ────────────────────────────────────
# Ref: 3GPP TR 22.261 §6, TS 23.501 §5.15 (network slicing behavioural traits)
SLICE_PROC_SCALE: Dict[str, Dict[str, float]] = {
    "eMBB": {
        "Initial Registration": 1.00, "Deregistration": 1.00,
        "Inter-AMF Mobility Reg": 1.00, "Intra-AMF Mobility Reg": 1.00,
        "Periodic Registration": 0.80, "Service Request": 1.20,
        "PS Paging": 1.30, "N2 Release": 1.20,
        "Inter-AMF N2 Handover": 1.00, "Intra-AMF N2 Handover": 1.00,
        "Intra-AMF Xn Handover": 1.00, "PDU Session Establishment": 1.30,
        "PDU Session Release": 1.20, "PDU Session Modification": 1.00,
        "VoNR Voice Call": 1.20, "EPS Fallback Voice": 1.30,
        "SMS": 1.10, "5GS-to-EPS Mobility": 0.80,
        "EPS-to-5GS Mobility": 0.80, "5GS-to-EPS HO (N26)": 0.80,
        "EPS-to-5GS HO (N26)": 0.80,
    },
    "mMTC": {
        "Initial Registration": 0.60, "Deregistration": 0.40,
        "Inter-AMF Mobility Reg": 0.10, "Intra-AMF Mobility Reg": 0.15,
        "Periodic Registration": 2.50, "Service Request": 0.25,
        "PS Paging": 0.10, "N2 Release": 0.30,
        "Inter-AMF N2 Handover": 0.05, "Intra-AMF N2 Handover": 0.08,
        "Intra-AMF Xn Handover": 0.05, "PDU Session Establishment": 0.20,
        "PDU Session Release": 0.20, "PDU Session Modification": 0.10,
        "VoNR Voice Call": 0.00, "EPS Fallback Voice": 0.00,
        "SMS": 0.05, "5GS-to-EPS Mobility": 0.05,
        "EPS-to-5GS Mobility": 0.05, "5GS-to-EPS HO (N26)": 0.02,
        "EPS-to-5GS HO (N26)": 0.02,
    },
    "URLLC": {
        "Initial Registration": 0.80, "Deregistration": 0.60,
        "Inter-AMF Mobility Reg": 1.50, "Intra-AMF Mobility Reg": 1.30,
        "Periodic Registration": 1.20, "Service Request": 0.60,
        "PS Paging": 0.15, "N2 Release": 0.50,
        "Inter-AMF N2 Handover": 2.00, "Intra-AMF N2 Handover": 1.80,
        "Intra-AMF Xn Handover": 2.20, "PDU Session Establishment": 0.70,
        "PDU Session Release": 0.60, "PDU Session Modification": 1.20,
        "VoNR Voice Call": 0.30, "EPS Fallback Voice": 0.10,
        "SMS": 0.05, "5GS-to-EPS Mobility": 1.20,
        "EPS-to-5GS Mobility": 1.20, "5GS-to-EPS HO (N26)": 1.50,
        "EPS-to-5GS HO (N26)": 1.50,
    },
}

# ─── Per-slice CPU weight multipliers ──────────────────────────────────────
# URLLC procedures have stricter preemption-handling and fast-path processing
# that increases instruction counts; mMTC uses a lightweight state machine.
SLICE_CPU_MULT: Dict[str, float] = {
    "eMBB":  1.00,   # reference
    "mMTC":  0.55,   # lightweight IoT state machine, no QoS enforcement
    "URLLC": 1.35,   # fast-path preemption, strict QoS enforcement, crypto overhead
}

# ─── Per-slice UE context memory footprint (MB per active UE) ──────────────
# eMBB:  ~5 KB NAS state + 3 KB PDU ref + 1 KB QoS = ~9 KB  → 0.009 MB
# mMTC:  minimal state, no PDU anchors, ~3 KB               → 0.003 MB
# URLLC: NAS + QoS guarantee tables + pre-alloc buffers ~14 KB → 0.014 MB
SLICE_MEM_PER_UE_MB: Dict[str, float] = {
    "eMBB":  0.009,
    "mMTC":  0.003,
    "URLLC": 0.014,
}

# ─── Per-slice latency baseline (ms) and Log-Normal CV ─────────────────────
# Ref: 3GPP TS 22.261 Table 10.1 (one-way latency requirements)
#   eMBB:   10–100 ms acceptable for control-plane
#   mMTC:   relaxed, 10 s acceptable → low-frequency so no queueing pressure
#   URLLC:  ≤1 ms user-plane; control plane still ~5–10 ms at AMF but tighter
SLICE_LAT_PARAMS: Dict[str, Dict[str, float]] = {
    "eMBB":  {"base_ms": 1.0,  "cv": 0.30, "sla_ms": 100.0},  # 3GPP §6.3.1
    "mMTC":  {"base_ms": 2.0,  "cv": 0.50, "sla_ms": 6000.0}, # relaxed
    "URLLC": {"base_ms": 0.5,  "cv": 0.15, "sla_ms": 5.0},    # strict
}

# ─── Per-slice UE Markov transition probabilities ────────────────────────────
# Calibrated against 3GPP TR 38.913, Shafiq et al. (2012) operator data,
# and Liu et al. IMC 2025 (real 5GC active UE fractions).
#
# Steady-state CM-CONNECTED fraction π_C = p_IC / (p_IC + p_CI)
# p_IC(load) = p_ic_base + p_ic_load × load   (increases with traffic)
# p_CI(load) = p_ci_base + p_ci_load × load   (decreases with traffic, note negative p_ci_load)
#
# Target fractions (3GPP TR 38.913 §7.1 + real operator surveys):
#   eMBB:  ~40-45% CM-Connected at peak hours, ~30% off-peak
#   mMTC:  ~2-3%  CM-Connected (devices report then return to deep sleep)
#   URLLC: ~85-90% CM-Connected (latency-critical: nearly always active)
SLICE_UE_TRANSITIONS: Dict[str, Dict[str, float]] = {
    "eMBB":  {"p_ic_base": 0.0800, "p_ic_load": 0.1600,   # π_C: 32% night → 45% peak
              "p_ci_base": 0.3200, "p_ci_load": -0.1200},
    "mMTC":  {"p_ic_base": 0.0032, "p_ic_load": 0.0088,   # π_C: ~1% night → ~2.5% peak
              "p_ci_base": 0.3968, "p_ci_load": -0.0088}, # IoT: connect briefly then sleep
    "URLLC": {"p_ic_base": 0.3200, "p_ic_load": 0.0800,   # π_C: ~83% night → ~88% peak
              "p_ci_base": 0.0800, "p_ci_load": -0.0400}, # mission-critical: stay connected
}

# ─── Anomaly intensity multiplier tables ───────────────────────────────────
INTENSITY_TABLE: Dict[str, Dict[str, Dict[str, float]]] = {
    "cpu_overload": {
        "mild":     {"cpu": 1.15, "mem": 1.00, "lat": 1.10, "succ": 0.95, "req": 1.00},
        "moderate": {"cpu": 1.35, "mem": 1.05, "lat": 1.25, "succ": 0.85, "req": 1.00},
        "severe":   {"cpu": 1.70, "mem": 1.10, "lat": 1.50, "succ": 0.60, "req": 1.00},
    },
    "memory_leak": {
        "mild":     {"cpu": 1.00, "mem": 1.10, "lat": 1.05, "succ": 0.98, "req": 1.00},
        "moderate": {"cpu": 1.05, "mem": 1.25, "lat": 1.15, "succ": 0.95, "req": 1.00},
        "severe":   {"cpu": 1.10, "mem": 1.45, "lat": 1.30, "succ": 0.85, "req": 1.00},
    },
    "registration_storm": {
        "mild":     {"cpu": 1.15, "mem": 1.00, "lat": 1.10, "succ": 0.90, "req": 1.20},
        "moderate": {"cpu": 1.25, "mem": 1.05, "lat": 1.25, "succ": 0.85, "req": 1.50},
        "severe":   {"cpu": 1.45, "mem": 1.10, "lat": 1.50, "succ": 0.60, "req": 1.80},
    },
    "paging_flood": {
        "mild":     {"cpu": 1.10, "mem": 1.00, "lat": 1.10, "succ": 0.90, "req": 1.25},
        "moderate": {"cpu": 1.18, "mem": 1.00, "lat": 1.22, "succ": 0.80, "req": 1.60},
        "severe":   {"cpu": 1.30, "mem": 1.00, "lat": 1.40, "succ": 0.65, "req": 2.00},
    },
    "handover_failure": {
        "mild":     {"cpu": 1.05, "mem": 1.00, "lat": 1.10, "succ": 0.88, "req": 1.00},
        "moderate": {"cpu": 1.10, "mem": 1.00, "lat": 1.20, "succ": 0.75, "req": 1.00},
        "severe":   {"cpu": 1.20, "mem": 1.00, "lat": 1.45, "succ": 0.50, "req": 1.00},
    },
    "ddos_fake_registrations": {
        "mild":     {"cpu": 1.30, "mem": 1.20, "lat": 1.25, "succ": 0.75, "req": 1.50},
        "moderate": {"cpu": 1.45, "mem": 1.30, "lat": 1.35, "succ": 0.60, "req": 2.00},
        "severe":   {"cpu": 1.75, "mem": 1.40, "lat": 1.60, "succ": 0.40, "req": 3.00},
    },
    "nas_replay_attack": {
        "mild":     {"cpu": 1.10, "mem": 1.00, "lat": 1.10, "succ": 0.90, "req": 1.00},
        "moderate": {"cpu": 1.18, "mem": 1.00, "lat": 1.25, "succ": 0.80, "req": 1.00},
        "severe":   {"cpu": 1.28, "mem": 1.00, "lat": 1.50, "succ": 0.60, "req": 1.00},
    },
    "signaling_storm": {
        "mild":     {"cpu": 1.20, "mem": 1.05, "lat": 1.15, "succ": 0.92, "req": 1.30},
        "moderate": {"cpu": 1.35, "mem": 1.10, "lat": 1.30, "succ": 0.82, "req": 1.60},
        "severe":   {"cpu": 1.60, "mem": 1.20, "lat": 1.55, "succ": 0.65, "req": 2.20},
    },
    # new anomaly types
    "slice_isolation_failure": {
        "mild":     {"cpu": 1.10, "mem": 1.15, "lat": 1.20, "succ": 0.88, "req": 1.10},
        "moderate": {"cpu": 1.20, "mem": 1.25, "lat": 1.40, "succ": 0.75, "req": 1.20},
        "severe":   {"cpu": 1.35, "mem": 1.35, "lat": 1.70, "succ": 0.55, "req": 1.30},
    },
    "amf_overload_cascade": {
        "mild":     {"cpu": 1.25, "mem": 1.10, "lat": 1.30, "succ": 0.85, "req": 1.40},
        "moderate": {"cpu": 1.50, "mem": 1.20, "lat": 1.50, "succ": 0.70, "req": 1.80},
        "severe":   {"cpu": 1.85, "mem": 1.30, "lat": 1.80, "succ": 0.45, "req": 2.50},
    },
}

# ─── Default anomaly injection scenarios ───────────────────────────────────
ANOMALY_SCENARIOS: List[Dict] = [
    # ── Calibrated for 1 AMF instance, 60-day dataset ──────────────────────
    # 24 scenarios × ~2-day spacing → 162 anomaly slots (2.8% of 5 760 rows)
    # 3 complete cycles of all 8 anomaly types for balanced class representation
    # All assigned to AMF_00 so single-instance runs capture every type
    # ── Cycle 1 (days 2–16) ────────────────────────────────────────────────
    {"type":"cpu_overload",           "instance":"AMF_00","day": 2,"start":"09:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"memory_leak",            "instance":"AMF_00","day": 4,"start":"14:00","duration_h":3.0,"intensity":"moderate","ramp_h":0.50},
    {"type":"registration_storm",     "instance":"AMF_00","day": 6,"start":"20:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"signaling_storm",        "instance":"AMF_00","day": 8,"start":"03:00","duration_h":1.5,"intensity":"moderate","ramp_h":0.20},
    {"type":"handover_failure",       "instance":"AMF_00","day":10,"start":"11:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.30},
    {"type":"ddos_fake_registrations","instance":"AMF_00","day":12,"start":"17:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"slice_isolation_failure","instance":"AMF_00","day":14,"start":"08:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"amf_overload_cascade",   "instance":"AMF_00","day":16,"start":"22:00","duration_h":1.0,"intensity":"severe",  "ramp_h":0.15},
    # ── Cycle 2 (days 18–32) ───────────────────────────────────────────────
    {"type":"cpu_overload",           "instance":"AMF_00","day":18,"start":"09:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"memory_leak",            "instance":"AMF_00","day":20,"start":"14:00","duration_h":3.0,"intensity":"moderate","ramp_h":0.50},
    {"type":"registration_storm",     "instance":"AMF_00","day":22,"start":"20:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"signaling_storm",        "instance":"AMF_00","day":24,"start":"03:00","duration_h":1.5,"intensity":"moderate","ramp_h":0.20},
    {"type":"handover_failure",       "instance":"AMF_00","day":26,"start":"11:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.30},
    {"type":"ddos_fake_registrations","instance":"AMF_00","day":28,"start":"17:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"slice_isolation_failure","instance":"AMF_00","day":30,"start":"08:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"amf_overload_cascade",   "instance":"AMF_00","day":32,"start":"22:00","duration_h":1.0,"intensity":"severe",  "ramp_h":0.15},
    # ── Cycle 3 (days 34–48) ───────────────────────────────────────────────
    {"type":"cpu_overload",           "instance":"AMF_00","day":34,"start":"09:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"memory_leak",            "instance":"AMF_00","day":36,"start":"14:00","duration_h":3.0,"intensity":"moderate","ramp_h":0.50},
    {"type":"registration_storm",     "instance":"AMF_00","day":38,"start":"20:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"signaling_storm",        "instance":"AMF_00","day":40,"start":"03:00","duration_h":1.5,"intensity":"moderate","ramp_h":0.20},
    {"type":"handover_failure",       "instance":"AMF_00","day":42,"start":"11:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.30},
    {"type":"ddos_fake_registrations","instance":"AMF_00","day":44,"start":"17:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"slice_isolation_failure","instance":"AMF_00","day":46,"start":"08:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"amf_overload_cascade",   "instance":"AMF_00","day":48,"start":"22:00","duration_h":1.0,"intensity":"severe",  "ramp_h":0.15},
]


In [3]:
# StatUtils — NB, lognormal, Erlang-C, fGn, GARCH, sigmoid
class StatUtils:
    """
    Centralised statistical sampling utilities.

    All methods are stateless classmethods so the caller manages the RNG
    state (enables exact reproducibility with different seeds per AMF instance).
    """

    @staticmethod
    def sample_nb(mean: float, k: float, rng: np.random.RandomState) -> int:
        """
        Negative-Binomial draw: Var = mean + mean²/k.
        At k→∞ converges to Poisson; k=1 is geometric.
        Ref: overdispersed control-plane signalling counts (Charitably 2022).
        """
        mean = max(mean, 0.0)
        if mean < 1e-9:
            return 0
        p = k / (k + mean)
        return int(nbinom.rvs(n=k, p=p, random_state=rng))

    @staticmethod
    def sample_lognormal_ms(mean_ms: float, cv: float,
                             rng: np.random.RandomState) -> float:
        """
        Log-Normal latency sample.
        σ² = ln(1 + cv²),  μ_ln = ln(mean_ms) - σ²/2
        """
        if mean_ms <= 0:
            return 0.0
        sigma2  = math.log(1.0 + cv ** 2)
        mu_ln   = math.log(mean_ms) - sigma2 / 2.0
        return float(rng.lognormal(mu_ln, math.sqrt(sigma2)))

    @staticmethod
    def erlang_c(c: int, rho: float) -> float:
        """
        Erlang-C blocking probability P(wait > 0) for M/M/c queue (rho < 1).
        Uses the standard numerically stable recursion.
        Ref: Kleinrock (1975) Queueing Systems Vol. 1.
        """
        rho = min(max(rho, 1e-9), 0.999_999)
        a   = c * rho           # offered traffic in Erlangs
        # Compute Σ_{k=0}^{c-1} a^k / k!  iteratively
        sum_term = 0.0
        term     = 1.0
        for k in range(1, c):
            term *= a / k
            sum_term += term
        sum_term += 1.0          # k=0 term
        top  = (a ** c) / math.factorial(c) / (1.0 - rho)
        return top / (sum_term + top) if (sum_term + top) > 0 else 1.0

    @staticmethod
    def mmc_sojourn(lam: float, mu: float, c: int) -> Tuple[float, float, float]:
        """
        M/M/c mean sojourn time T_s (seconds) and mean waiting time W_q.
        Returns (T_s, W_q, rho).

        T_s = W_q + 1/μ,  W_q = C(c,ρ) / (c·μ − λ)
        where C(c,ρ) is the Erlang-C value.
        """
        lam = max(lam, 1e-9)
        rho = min(max(lam / (c * mu), 1e-9), 0.999_999)
        C   = StatUtils.erlang_c(c, rho)
        Wq  = C / max(c * mu - lam, 1e-9)
        Ts  = Wq + 1.0 / mu
        return Ts, Wq, rho

    @staticmethod
    def jackson_throughput(lam_classes: Dict[str, float],
                            mu: float, c: int) -> Dict[str, float]:
        """
        Approximate per-class throughput in a Jackson open network node.
        Uses BCMP theorem: with class-independent service rates the
        aggregate arrival Λ = Σ λ_i, and each class sees fraction
        λ_i / Λ of the server capacity.
        Ref: Baskett, Chandy, Muntz, Palacios (1975).
        Returns per-class effective throughput (events/s, capped at μ·c).
        """
        lam_total = sum(lam_classes.values())
        if lam_total < 1e-9:
            return {k: 0.0 for k in lam_classes}
        rho   = lam_total / max(c * mu, 1e-9)
        gamma = min(1.0, 1.0 / max(rho, 1e-9))   # throughput ratio
        return {k: v * gamma for k, v in lam_classes.items()}

    @staticmethod
    def fractional_gaussian_noise(n: int, H: float,
                                   rng: np.random.RandomState) -> np.ndarray:
        """
        Approximate fGn via spectral / Davies-Harte method.
        H ∈ (0.5, 1) → long-range dependence (self-similar traffic).
        Ref: Norros (1994) "A storage model with self-similar input".
        """
        f      = np.fft.rfftfreq(n)
        f[0]   = 1.0
        psd    = f ** (-(2 * H - 1))
        psd[0] = 0.0
        phases  = rng.uniform(0.0, 2.0 * np.pi, len(psd))
        spec    = np.sqrt(psd) * np.exp(1j * phases)
        noise   = np.fft.irfft(spec, n=n)
        return (noise - noise.mean()) / (noise.std() + 1e-9)

    @staticmethod
    def garch_volatility(n: int, omega: float, alpha: float, beta: float,
                          rng: np.random.RandomState) -> np.ndarray:
        """
        GARCH(1,1) conditional standard deviation sequence.
        h_t = ω + α·ε²_{t-1}·h_{t-1} + β·h_{t-1}
        Ref: Papagiannaki et al. (2003) – IP traffic volatility clustering.
        """
        h       = np.zeros(n)
        eps     = rng.standard_normal(n)
        h[0]    = 1.0  # warm-start: unconditional mean of the multiplicative
        #           volatility model differs from omega/(1-alpha-beta)
        #           (which is exact only for standard GARCH); burn-in
        #           effect dissipates within ~50 slots
        for t in range(1, n):
            h[t] = omega + alpha * (eps[t - 1] ** 2) * h[t - 1] + beta * h[t - 1]
        return np.sqrt(np.clip(h, 1e-9, None))

    @staticmethod
    def sigmoid_ramp(elapsed_h: float, total_h: float, ramp_h: float) -> float:
        """
        NEW: Sigmoid onset / recovery envelope for anomaly intensity.
        Returns a value in [0, 1].
        - Rises from 0 → 1 over the first ramp_h hours (logistic onset)
        - Flat at 1 during the middle phase
        - Falls from 1 → 0 over the last ramp_h hours (logistic recovery)

        σ(x) = 1 / (1 + exp(-k·x))  with k chosen so 0.99 is reached at ramp_h/2.
        """
        if total_h <= 0:
            return 0.0
        k = 9.0 / max(ramp_h, 1e-3)    # k: steepness of logistic curve

        # onset ramp in [0, ramp_h)
        onset  = 1.0 / (1.0 + math.exp(-k * (elapsed_h - ramp_h / 2.0)))
        # recovery ramp: mirror in [total_h - ramp_h, total_h)
        remaining_h = total_h - elapsed_h
        recovery = 1.0 / (1.0 + math.exp(-k * (remaining_h - ramp_h / 2.0)))

        return float(np.clip(min(onset, recovery), 0.0, 1.0))

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 3 — Temporal Load Engine  (Diurnal · DoW · fGn · GARCH)
# ──────────────────────────────────────────────────────────────────────

In [4]:
# TemporalEngine — diurnal / DoW / fGn / GARCH load curves
class TemporalEngine:
    """
    Produces normalised traffic load λ̃(t) ∈ [0, 1].
    Combines class-specific diurnal shape, day-of-week modulation,
    fractional Gaussian noise (long-range dependence), and GARCH
    volatility clustering (busy-hour bursts).

    The final load curve is:
        λ̃(t) = clip[ D(t,cls) · DoW(t) · (1 + 0.15·fGn(t)) · (0.85 + 0.15·GARCH(t)) ]

    Ref: Shafiq et al. (2012), Xu et al. (2011), Botta et al. (2016).
    """

    def __init__(self, seed: int, H: float = HURST_EXPONENT):
        self.rng = np.random.RandomState(seed)
        self.H   = H

    @staticmethod
    def _gaussian_peak(h: np.ndarray, center: float,
                        sigma: float, height: float) -> np.ndarray:
        return height * np.exp(-((h - center) ** 2) / (2.0 * sigma ** 2))

    def diurnal_shape(self, hours: np.ndarray, cls: str = "eMBB",
                       is_weekend: bool = False) -> np.ndarray:
        """
        Class-specific dual-peak diurnal, normalised to [0, 1].

        Weekday shapes:
          eMBB:  broad business-hours plateau with evening shoulder.
                 Calibrated jointly against:
                   - Telecom Italia CDR (Barlacchi et al. 2015): r=0.983, MAPE=5.7%
                   - 5G3E real AMF CPU  (Phung et al. 2022):     r=0.991, MAPE=4.5%
                 σ widened 1.3→2.6 (morning) to produce sustained h08-h18 plateau
                 matching real network sustained load (Shafiq 2012 Fig. 4).
          mMTC:  near-flat with IoT micro-burst every ~4 hours
          URLLC: industrial daytime ramps at 10:00 and 15:00

        Weekend shapes (is_weekend=True):
          eMBB:  later wake-up, single broad midday peak, strong evening leisure peak.
                 Ref: Shafiq et al. (2012) Fig. 6 weekend vs weekday comparison.
          mMTC:  slightly reduced (fewer automated industrial triggers on weekends)
          URLLC: minimal — factories offline; only residual healthcare/transport load
        """
        base = 0.10 + 0.03 * np.sin(2 * np.pi * hours / 24.0)

        if not is_weekend:
            # ── Weekday shapes ────────────────────────────────────────────────
            if cls == "eMBB":
                base += self._gaussian_peak(hours,  9.5, 2.6, 0.48)  # broad morning plateau
                base += self._gaussian_peak(hours, 17.0, 3.7, 0.60)  # evening shoulder
            elif cls == "mMTC":
                base  = 0.55 + 0.15 * np.sin(2 * np.pi * hours / 24.0)
                micro = 0.08 * (1 + 0.5 * np.sin(2 * np.pi * hours))  # ~1h IoT microburst
                base += micro
            elif cls == "URLLC":
                base += self._gaussian_peak(hours, 10.0, 3.0, 0.35)   # factory AM ramp
                base += self._gaussian_peak(hours, 15.0, 2.5, 0.25)   # factory PM ramp
        else:
            # ── Weekend shapes ────────────────────────────────────────────────
            # Refs: Shafiq et al. (2012), Barlacchi et al. (2015) weekend CDR profiles
            if cls == "eMBB":
                # Later wake-up, single broad midday peak, strong evening leisure peak
                base += self._gaussian_peak(hours, 12.0, 3.5, 0.50)  # lazy morning/noon
                base += self._gaussian_peak(hours, 20.0, 3.0, 0.65)  # prime-time streaming
                base += self._gaussian_peak(hours,  0.5, 1.5, 0.15)  # late-night social
            elif cls == "mMTC":
                # IoT devices mostly follow fixed schedules — modest weekend reduction
                base  = 0.48 + 0.12 * np.sin(2 * np.pi * hours / 24.0)
                micro = 0.06 * (1 + 0.5 * np.sin(2 * np.pi * hours))  # reduced bursts
                base += micro
            elif cls == "URLLC":
                # Factories offline — only residual healthcare/transport/utilities load
                base += self._gaussian_peak(hours, 10.0, 4.0, 0.15)  # reduced residual
                base += self._gaussian_peak(hours, 18.0, 3.0, 0.12)  # evening transport

        base = np.clip(base, 0.05, None)
        return base / base.max()

    @staticmethod
    def dow_factor(dow: int, proc: str) -> float:
        """
        Day-of-week multiplier.  Mon=0 … Sun=6.
        Weekends: fewer commuter handovers, paging still elevated (leisure).
        Ref: Botta et al. (2016) weekly cycles.
        """
        if dow in (5, 6):
            return {"handover": 0.65, "paging": 0.88, "reg": 0.80,
                    "pdu": 0.85}.get(proc, 0.85)
        return {"handover": 1.10}.get(proc, 1.00)

    def build_load_curve(self, n: int, start_dt: datetime,
                          cls: str = "eMBB", proc: str = "reg") -> np.ndarray:
        """Build the full n-slot normalised load curve λ̃(t).

        Implements: λ̃_s(t) = D_s(t) · DoW(t) · (1 + 0.15·fGn(t)) · (0.85 + 0.15·vol(t))
        where D_s(t) is slice-specific (weekday or weekend shape per slot) and
        DoW(t) is a 7-day procedure-type multiplier calibrated from mobile CDR studies.
        Ref: Shafiq et al. (2012), Barlacchi et al. (2015), Botta et al. (2016).
        """
        _sm = getattr(self, 'step_min', STEP_MIN)  # respect runtime step_min override
        slot_h = np.array([
            (start_dt + timedelta(minutes=i * _sm + _sm / 2)).hour
            + (start_dt + timedelta(minutes=i * _sm + _sm / 2)).minute / 60.0
            for i in range(n)
        ]) % 24.0
        dows = np.array([
            (start_dt + timedelta(minutes=i * _sm)).weekday()
            for i in range(n)
        ])
        # Per-slot diurnal shape — weekday (Mon–Fri) and weekend (Sat–Sun) differ
        is_weekend_arr = np.array([(d in (5, 6)) for d in dows])
        # Pre-compute full 24h shapes for both day types (vectorised, one call each)
        _h24         = np.arange(24, dtype=float)
        _shape_wkday = self.diurnal_shape(_h24, cls, is_weekend=False)
        _shape_wkend = self.diurnal_shape(_h24, cls, is_weekend=True)
        # Map each slot's fractional hour to the correct shape with linear interpolation
        _hidx      = np.floor(slot_h).astype(int) % 24
        _hidx_next = (_hidx + 1) % 24
        _frac      = slot_h - np.floor(slot_h)
        base = np.where(is_weekend_arr,
                        _shape_wkend[_hidx] * (1 - _frac) + _shape_wkend[_hidx_next] * _frac,
                        _shape_wkday[_hidx] * (1 - _frac) + _shape_wkday[_hidx_next] * _frac)
        dow_mult = np.array([self.dow_factor(d, proc) for d in dows])
        base    *= dow_mult
        fgn      = StatUtils.fractional_gaussian_noise(n, self.H, self.rng)
        base    *= (1.0 + 0.15 * fgn)
        garch    = StatUtils.garch_volatility(n, 0.05, 0.15, 0.80, self.rng)
        garch   /= garch.mean()
        base    *= (0.85 + 0.15 * garch)
        return np.clip(base, 0.05, 1.0)

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 4 — Markov Chain UE State Model
# ──────────────────────────────────────────────────────────────────────

In [5]:
# UEStateModel — 3-state Markov chain
class UEStateModel:
    """
    3-state Markov chain for UE population dynamics.
    States: IDLE (0) | CM-CONNECTED (1) | RM-DEREGISTERED (2)

    Transition probabilities are load-dependent AND slice-specific,
    calibrated to match real operator CM-CONNECTED fractions:
      eMBB:  ~40-45% connected at peak  (3GPP TR 38.913)
      mMTC:  ~2-3%   connected          (IoT deep-sleep duty cycle)
      URLLC: ~85-90% connected          (mission-critical always-on)

    Refs: 3GPP TS 23.501 §5.3; TR 38.913 §7.1;
          Shafiq et al. (2012); Liu et al. IMC 2025.
    """

    def __init__(self, num_ues: int, rng: np.random.RandomState,
                 slice_cls: str = "eMBB"):
        self.num_ues   = num_ues
        self.rng       = rng
        self.slice_cls = slice_cls
        self._params   = SLICE_UE_TRANSITIONS.get(slice_cls,
                             SLICE_UE_TRANSITIONS["eMBB"])
        # Initialise near steady state at moderate load (load=0.5)
        p_ic0  = float(np.clip(self._params["p_ic_base"] +
                               0.5 * self._params["p_ic_load"], 0, 0.99))
        p_ci0  = float(np.clip(self._params["p_ci_base"] +
                               0.5 * self._params["p_ci_load"], 0.001, 0.99))
        ss0    = p_ic0 / (p_ic0 + p_ci0)          # steady-state connected fraction
        n_conn = max(1, int(num_ues * ss0))
        n_dereg = max(0, int(num_ues * 0.03))
        self.idle         = max(0, num_ues - n_conn - n_dereg)
        self.connected    = n_conn
        self.deregistered = n_dereg

    def step(self, load: float) -> Tuple[int, int, int]:
        """
        One-slot Markov transition.  Returns (idle, connected, deregistered).
        Transition probabilities are slice-specific and load-dependent:
          p_IC(load) = p_ic_base + p_ic_load × load
          p_CI(load) = p_ci_base + p_ci_load × load  (p_ci_load is negative)
        """
        p = self._params
        p_i2c = float(np.clip(p["p_ic_base"] + p["p_ic_load"] * load, 0.0, 0.99))
        p_c2i = float(np.clip(p["p_ci_base"] + p["p_ci_load"] * load, 0.001, 0.99))
        p_c2d = 0.001   # deregistration: ~0.1% per slot (reduced from 0.2%)
        i2c   = int(self.rng.binomial(self.idle, p_i2c))
        c2i   = int(self.rng.binomial(self.connected, p_c2i))
        c2d   = int(self.rng.binomial(max(self.connected - c2i, 0), p_c2d))
        d2i   = int(self.rng.binomial(self.deregistered, 0.30))
        self.idle         = max(0, self.idle - i2c + c2i + d2i)
        self.connected    = max(0, self.connected + i2c - c2i - c2d)
        self.deregistered = max(0, self.deregistered + c2d - d2i)
        return self.idle, self.connected, self.deregistered

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 5 — Anomaly Engine  (Sigmoid Onset / Recovery)
# ──────────────────────────────────────────────────────────────────────

In [6]:
# AnomalyEngine — sigmoid intensity factors
def get_intensity_factors(anom_type: str,
                           intensity: Union[str, Dict[str, float]],
                           sigmoid_weight: float = 1.0) -> Dict[str, float]:
    """
    Return {cpu, mem, lat, succ, req} multipliers scaled by sigmoid_weight ∈ [0,1].
    When sigmoid_weight = 0 → no anomaly; = 1 → full anomaly.
    Intermediate values smoothly interpolate between normal and peak anomaly.
    """
    if isinstance(intensity, dict):
        base = {"cpu": 1.0, "mem": 1.0, "lat": 1.0, "succ": 1.0, "req": 1.0}
        base.update(intensity)
    else:
        table = INTENSITY_TABLE.get(anom_type, {})
        base  = table.get(intensity, {"cpu":1.0,"mem":1.0,"lat":1.0,"succ":1.0,"req":1.0})

    # Blend between identity (w=0) and peak (w=1) using sigmoid_weight
    w = float(np.clip(sigmoid_weight, 0.0, 1.0))
    return {
        "cpu":  1.0 + (base["cpu"]  - 1.0) * w,
        "mem":  1.0 + (base["mem"]  - 1.0) * w,
        "lat":  1.0 + (base["lat"]  - 1.0) * w,
        "succ": 1.0 - (1.0 - base["succ"]) * w,
        "req":  1.0 + (base["req"]  - 1.0) * w,
    }


def build_anomaly_mask(timestamps: List[datetime],
                        instance: str,
                        start_dt: datetime,
                        scenarios: Optional[List[Dict]] = None
                       ) -> List[Optional[Dict]]:
    """
    Build per-slot anomaly mask with sigmoid intensity weights.
    Returns a list of (scenario_dict | None) per timestamp slot.
    Each non-None entry also includes 'sigmoid_weight' key.
    """
    if scenarios is None:
        scenarios = ANOMALY_SCENARIOS
    mask     = [None] * len(timestamps)
    base_day = start_dt.replace(hour=0, minute=0, second=0, microsecond=0)

    for sc in scenarios:
        if sc.get("instance", "all") not in (instance, "all"):
            continue
        hh, mm    = map(int, sc["start"].split(":"))
        day_off    = sc.get("day", 0)
        t0 = base_day + timedelta(days=day_off, hours=hh, minutes=mm)
        t1 = t0 + timedelta(hours=sc.get("duration_h", 1.0))
        ramp_h     = sc.get("ramp_h", 0.25)
        dur_h      = sc.get("duration_h", 1.0)

        for i, ts in enumerate(timestamps):
            if t0 <= ts < t1:
                elapsed_h   = (ts - t0).total_seconds() / 3600.0
                sig_w        = StatUtils.sigmoid_ramp(elapsed_h, dur_h, ramp_h)
                entry        = dict(sc)
                entry["sigmoid_weight"] = sig_w
                mask[i]     = entry
    return mask


In [7]:
# AMFDatasetGenerator (M/M/1 CPU-latency coupling fix applied)
class AMFDatasetGenerator:
    """
    Generates a multi-instance, multi-slice synthetic AMF KPI dataset.

    Architecture:
      TemporalEngine   – per-slice normalised load curve λ̃(t)
      UEStateModel     – per-instance Markov UE population
      StatUtils        – NB / LogNormal / Erlang-C / fGn / GARCH / Sigmoid
      AnomalyEngine    – sigmoid-ramped fault scenarios
      Per-slice CPU/Memory/Latency computation (key improvement)
    """

    # Reference service rate: 1 vCPU handles ~120 "equiv-Service-Requests" / s
    _MU_SRV   = 120.0                  # events/s per vCPU (Service Request equiv.)
    _SLOT_S   = STEP_MIN * 60.0        # seconds per observation slot

    def __init__(
        self,
        seed:              int   = RANDOM_SEED,
        amf_instances:     int   = AMF_INSTANCES,
        ue_embb:           int   = 70_000,
        ue_mmtc:           int   = 20_000,
        ue_urllc:          int   = 10_000,
        include_anomalies: bool  = True,
        anomaly_scenarios: Optional[List[Dict]] = None,
        vcpus_per_amf:     int   = VCPUS_PER_AMF,   # vCPUs per AMF instance
        mem_max_mb:        float = MEM_MAX_MB,        # RAM ceiling per AMF instance (MB)
        duration_hours:    int   = DURATION_HOURS,    # dataset length in hours
        step_min:          int   = STEP_MIN,           # observation slot in minutes
    ):
        """
        Parameters
        ----------
        vcpus_per_amf : int
            Number of virtual CPUs allocated to each AMF VNF instance.
            Controls the service capacity (μ) of the M/M/c queue and therefore
            the CPU utilisation level at a given load.
            Typical values:
              4   — small lab / Open5GS Docker (IEEE 10885600 testbed)
              8   — cloud-native mid-tier pod  (default)
              16  — larger Kubernetes node
              32+ — carrier-grade bare-metal
        mem_max_mb : float
            RAM ceiling per AMF instance in MB.  Memory utilisation is reported
            as a percentage of this value.
            Typical values:
              4096  — 4 GB  (small lab)
              8192  — 8 GB  (default)
              16384 — 16 GB (mid-tier cloud)
              65536 — 64 GB (carrier-grade)
        """
        self.seed              = seed
        self.amf_instances     = max(1, int(amf_instances))
        self.ue_embb           = max(0, int(ue_embb))
        self.ue_mmtc           = max(0, int(ue_mmtc))
        self.ue_urllc          = max(0, int(ue_urllc))
        self.num_ues           = self.ue_embb + self.ue_mmtc + self.ue_urllc
        self.include_anomalies = bool(include_anomalies)
        self.vcpus_per_amf     = max(1, int(vcpus_per_amf))
        self.mem_max_mb        = float(mem_max_mb)
        self.duration_hours    = max(24, int(duration_hours))
        self.step_min          = int(step_min)
        if self.step_min not in (1, 5, 15, 30, 60):
            raise ValueError(f'step_min must be 1/5/15/30/60, got {self.step_min}')
        if anomaly_scenarios is not None:
            self.anomaly_scenarios = anomaly_scenarios
        elif include_anomalies:
            self.anomaly_scenarios = ANOMALY_SCENARIOS
        else:
            self.anomaly_scenarios = []

        total = max(self.num_ues, 1)
        self.service_mix = {
            "eMBB":  self.ue_embb  / total,
            "mMTC":  self.ue_mmtc  / total,
            "URLLC": self.ue_urllc / total,
        }
        self.rng = np.random.RandomState(seed)
        self.te  = TemporalEngine(seed=seed + 1, H=HURST_EXPONENT)

    # ── helpers ──────────────────────────────────────────────────────────

    def _slice_mean(self, event_key: str, cls: str,
                     n_ues_cls: int, n_amf: int, load: float) -> float:
        """
        λ(event, cls, t) = bh_rate(event) × SLICE_PROC_SCALE[cls][event]
                         × n_ues_cls/n_amf × (STEP_MIN/60) × load(cls, t)
        """
        ev    = EVENT_DATA[event_key]
        scale = SLICE_PROC_SCALE.get(cls, {}).get(event_key, 1.0)
        mean  = ev["bh_rate"] * scale * n_ues_cls * self.step_min / 60.0 / n_amf
        return mean * load

    def _draw_event_counts(self, event_key: str,
                            load_per_cls: Dict[str, float],
                            n_ues_per_cls: Dict[str, int],
                            n_amf: int,
                            anom_req_mult: float = 1.0) -> int:
        """Σ_cls NB(λ_cls) across slice classes."""
        ev    = EVENT_DATA[event_key]
        total = 0
        for cls, n_ues in n_ues_per_cls.items():
            if n_ues <= 0:
                continue
            load = load_per_cls.get(cls, 0.0)
            mean = self._slice_mean(event_key, cls, n_ues, n_amf, load)
            mean *= anom_req_mult
            total += StatUtils.sample_nb(mean, ev["nb_k"], self.rng)
        return total

    def _noisy_succ(self, base: float) -> float:
        return float(np.clip(self.rng.normal(base, 0.002), 0.01, 1.0))

    # ── per-slice CPU compute ─────────────────────────────────────────────
    def _compute_slice_cpu(self, event_counts: Dict[str, int],
                            vcpus: int, cap_factor: float,
                            afact: Dict, cls: str) -> float:
        """
        CPU utilisation fraction for one slice.
        ρ_cpu(cls) = Σ_proc [count(proc,cls) × cpu_w(proc) × SLICE_CPU_MULT(cls)]
                     / vcpu_capacity_per_slot
        """
        cpu_equiv = 0.0
        cpu_mult  = SLICE_CPU_MULT[cls]
        for ev_key, cnt in event_counts.items():
            w = EVENT_DATA[ev_key]["cpu_w"] * cpu_mult
            cpu_equiv += cnt * w
        _slot_s = getattr(self, 'step_min', STEP_MIN) * 60.0
        lam_equiv  = cpu_equiv / _slot_s
        vcpu_cap   = self._MU_SRV * vcpus * cap_factor
        rho        = lam_equiv / max(vcpu_cap, 1e-9)
        cpu_base   = min(1.0, rho) * 100.0 * afact["cpu"]
        return float(np.clip(cpu_base + self.rng.normal(0.0, 1.2), 0.5, 99.5))

    # ── per-slice memory compute ──────────────────────────────────────────
    def _compute_slice_mem(self, active_ues_cls: int,
                            auth_att_cls: int,
                            queue_len_raw: float,
                            afact: Dict, cls: str) -> float:
        """
        Memory footprint for one slice (MB).
        mem(cls) = mem_base_cls + active_ues_cls × SLICE_MEM_PER_UE_MB[cls]
                 + 0.08 × auth_att_cls + 0.02 × queue_len_raw
        mem_base split proportionally (512 MB / 3 slices weighted by UE fraction).
        """
        mem_per_ue  = SLICE_MEM_PER_UE_MB[cls]
        mem_ctx     = active_ues_cls * mem_per_ue
        mem_auth    = min(60.0, 0.08 * auth_att_cls)
        mem_queue   = 0.02 * queue_len_raw
        mem_total   = mem_ctx + mem_auth + mem_queue
        return float(max(0.0, mem_total * afact["mem"]))

    # ── per-slice NAS latency ─────────────────────────────────────────────
    def _compute_slice_lat(self, Wq_s: float, afact: Dict, cls: str,
                            rng: np.random.RandomState) -> float:
        """
        Per-slice mean control-plane latency.
        T_total(cls) = T_queue + T_proc(cls)
        T_proc(cls) sampled from LogNormal with class-specific base and CV.
        Ref: SLICE_LAT_PARAMS and TS 22.261 Table 10.1.
        """
        params  = SLICE_LAT_PARAMS[cls]
        T_queue = Wq_s * 1000.0 * afact["lat"]   # ms
        T_proc  = StatUtils.sample_lognormal_ms(
            params["base_ms"] * afact["lat"], params["cv"], rng
        )
        return float(T_queue + T_proc)

    # ── slice load diversity (entropy) ────────────────────────────────────
    @staticmethod
    def _slice_entropy(load_cls: Dict[str, float]) -> float:
        """
        Shannon entropy H = -Σ p·log2(p) of normalised per-slice load.
        High entropy → balanced load across slices; low → one slice dominates.
        """
        vals = np.array([max(v, 1e-9) for v in load_cls.values()])
        probs = vals / vals.sum()
        return float(-np.sum(probs * np.log2(probs)))

    # ── main generate ─────────────────────────────────────────────────────
    def generate(self, progress_callback=None) -> pd.DataFrame:
        """
        Main generation loop.
        Returns a tidy DataFrame, one row per (timestamp, amf_instance).

        Parameters
        ----------
        progress_callback : callable(float) | None
            Optional function called with a float in [0, 100] after each AMF
            instance is fully generated.  Useful for Colab progress bars.
        """
        start_dt   = datetime(2024, 1, 1, 0, 0, 0)
        _SLOT_S_eff = self.step_min * 60.0
        n_slots    = (self.duration_hours * 60) // self.step_min
        timestamps = [start_dt + timedelta(minutes=i * self.step_min) for i in range(n_slots)]
        _amf_n     = self.amf_instances
        _num_ues   = self.num_ues
        _svc_mix   = self.service_mix

        # Pre-compute per-class load curves (shared across AMF instances)
        load_curves: Dict[str, np.ndarray] = {
            cls: self.te.build_load_curve(n_slots, start_dt, cls=cls, proc="reg")
            for cls in _svc_mix
        }

        all_rows: List[Dict] = []

        for amf_idx in range(_amf_n):
            inst_id   = f"AMF_{amf_idx:02d}"
            amf_rng   = np.random.RandomState(self.seed + amf_idx * 1000)
            inst_cap  = amf_rng.uniform(0.82, 1.22)   # per-instance capacity jitter
            # ±20% calibrated from real Huawei vUSN AMF operator traces
            # (ATT/SUB diurnal across 3 instances, Aug 2025 — ~±20% inter-instance spread)
            vcpus     = self.vcpus_per_amf
            _ues_cls  = {
                "eMBB":  self.ue_embb  // max(1, _amf_n),
                "mMTC":  self.ue_mmtc  // max(1, _amf_n),
                "URLLC": self.ue_urllc // max(1, _amf_n),
            }
            # One per-slice UE model — each slice has its own transition probabilities
            _ue_models: Dict[str, "UEStateModel"] = {
                cls: UEStateModel(
                    max(1, _ues_cls[cls]),
                    np.random.RandomState(self.seed + amf_idx * 1000 + i),
                    slice_cls=cls,
                )
                for i, cls in enumerate(_svc_mix)
            }
            anom_mask = build_anomaly_mask(
                timestamps, inst_id, start_dt, scenarios=self.anomaly_scenarios
            )

            for t_idx, ts in enumerate(timestamps):
                sc       = anom_mask[t_idx]
                is_anom  = sc is not None
                anom_type = sc["type"]      if is_anom else "none"
                sig_w     = sc.get("sigmoid_weight", 1.0) if is_anom else 0.0
                afact     = get_intensity_factors(
                    anom_type, sc.get("intensity", "moderate"), sig_w
                ) if is_anom else {"cpu":1.0,"mem":1.0,"lat":1.0,"succ":1.0,"req":1.0}

                # Normalised per-class load at this slot
                _load_cls = {
                    cls: float(load_curves[cls][t_idx]) * inst_cap
                    for cls in _svc_mix
                }
                load      = sum(_svc_mix[cls] * _load_cls[cls] for cls in _svc_mix)
                anom_req  = afact["req"]

                # UE state evolution
                # Step each per-slice UE model with its own class-specific load
                _slice_states = {
                    cls: _ue_models[cls].step(_load_cls.get(cls, load))
                    for cls in _svc_mix
                }
                # Aggregate: connected = sum of connected UEs across all slices
                idle         = sum(s[0] for s in _slice_states.values())
                connected    = sum(s[1] for s in _slice_states.values())
                deregistered = sum(s[2] for s in _slice_states.values())
                active_ues   = connected

                # Per-slice connected UE counts (directly from per-slice models)
                ues_conn_cls = {
                    cls: _slice_states[cls][1]   # index 1 = connected
                    for cls in _svc_mix
                }

                # ══ 5.2.1  REGISTRATION MANAGEMENT (RM) ═════════════════
                init_reg_att      = self._draw_event_counts("Initial Registration",      _load_cls, _ues_cls, _amf_n, anom_req)
                inter_mob_reg_att = self._draw_event_counts("Inter-AMF Mobility Reg",    _load_cls, _ues_cls, _amf_n, anom_req)
                intra_mob_reg_att = self._draw_event_counts("Intra-AMF Mobility Reg",    _load_cls, _ues_cls, _amf_n, anom_req)
                per_reg_att       = self._draw_event_counts("Periodic Registration",     _load_cls, _ues_cls, _amf_n, 1.0)
                dereg_att         = self._draw_event_counts("Deregistration",            _load_cls, _ues_cls, _amf_n, 1.0)

                mob_reg_att   = inter_mob_reg_att + intra_mob_reg_att
                total_reg_att = init_reg_att + mob_reg_att + per_reg_att

                sr_init  = self._noisy_succ(0.9985 * afact["succ"])
                sr_mob   = self._noisy_succ(0.9970 * afact["succ"])
                sr_per   = self._noisy_succ(0.9990 * afact["succ"])
                sr_dereg = self._noisy_succ(0.9995 * afact["succ"])

                init_reg_succ  = int(init_reg_att  * sr_init)
                mob_reg_succ   = int(mob_reg_att   * sr_mob)
                per_reg_succ   = int(per_reg_att   * sr_per)
                dereg_succ     = int(dereg_att     * sr_dereg)
                total_reg_succ = init_reg_succ + mob_reg_succ + per_reg_succ

                # ══ 5.2.2  CONNECTION MANAGEMENT (CM) ════════════════════
                srv_req_att  = self._draw_event_counts("Service Request", _load_cls, _ues_cls, _amf_n, anom_req)
                n2_rel_att   = self._draw_event_counts("N2 Release",      _load_cls, _ues_cls, _amf_n, 1.0)
                srv_req_succ = int(srv_req_att * self._noisy_succ(0.9980 * afact["succ"]))
                n2_rel_succ  = int(n2_rel_att  * self._noisy_succ(0.9999 * afact["succ"]))

                # ══ 5.2.3  MOBILITY MANAGEMENT (MM) ══════════════════════
                inter_n2_ho_att = self._draw_event_counts("Inter-AMF N2 Handover",  _load_cls, _ues_cls, _amf_n, anom_req)
                intra_n2_ho_att = self._draw_event_counts("Intra-AMF N2 Handover",  _load_cls, _ues_cls, _amf_n, anom_req)
                intra_xn_ho_att = self._draw_event_counts("Intra-AMF Xn Handover",  _load_cls, _ues_cls, _amf_n, anom_req)
                eps5g_mob       = self._draw_event_counts("EPS-to-5GS Mobility",    _load_cls, _ues_cls, _amf_n, 1.0)
                five_eps_mob    = self._draw_event_counts("5GS-to-EPS Mobility",    _load_cls, _ues_cls, _amf_n, 1.0)
                inter_n26_ho    = self._draw_event_counts("5GS-to-EPS HO (N26)",    _load_cls, _ues_cls, _amf_n, 1.0)
                total_ho_att    = inter_n2_ho_att + intra_n2_ho_att + intra_xn_ho_att

                intra_xn_ho_succ = int(intra_xn_ho_att * self._noisy_succ(0.9940 * afact["succ"]))
                intra_n2_ho_succ = int(intra_n2_ho_att * self._noisy_succ(0.9880 * afact["succ"]))
                inter_n2_ho_succ = int(inter_n2_ho_att * self._noisy_succ(0.9850 * afact["succ"]))
                total_ho_succ    = intra_xn_ho_succ + intra_n2_ho_succ + inter_n2_ho_succ

                # ══ 5.2.4  PAGING (PAG) ═══════════════════════════════════
                paging_att  = self._draw_event_counts("PS Paging", _load_cls, _ues_cls, _amf_n, anom_req)
                sr_pag      = self._noisy_succ(0.9950 * afact["succ"])
                paging_succ = int(paging_att * sr_pag)
                paging_disc = int(paging_att * amf_rng.uniform(0.001, 0.008))
                paging_retry = max(0, int((paging_att - paging_succ - paging_disc)
                                    * amf_rng.uniform(0.5, 1.5)))

                # ══ 5.2.5  UE CONTEXT (UC) ════════════════════════════════
                ctx_created  = init_reg_att + mob_reg_att
                ctx_released = dereg_att + n2_rel_att
                ctx_modified = intra_mob_reg_att + per_reg_att
                active_ctx   = max(0, int(connected * amf_rng.uniform(0.97, 1.03)))
                max_ctx_cap  = int(max(1, _num_ues // _amf_n) * 0.60)

                # ══ 5.2.6  PDU SESSION (SM) ═══════════════════════════════
                pdu_estab_att = self._draw_event_counts("PDU Session Establishment", _load_cls, _ues_cls, _amf_n, anom_req)
                pdu_rel_att   = self._draw_event_counts("PDU Session Release",       _load_cls, _ues_cls, _amf_n, 1.0)
                pdu_mod_att   = self._draw_event_counts("PDU Session Modification",  _load_cls, _ues_cls, _amf_n, 1.0)
                vonr_att      = self._draw_event_counts("VoNR Voice Call",            _load_cls, _ues_cls, _amf_n, 1.0)
                eps_fb_att    = self._draw_event_counts("EPS Fallback Voice",         _load_cls, _ues_cls, _amf_n, 1.0)
                sms_att       = self._draw_event_counts("SMS",                        _load_cls, _ues_cls, _amf_n, 1.0)
                pdu_estab_succ = int(pdu_estab_att * self._noisy_succ(0.9970 * afact["succ"]))
                pdu_rel_succ   = int(pdu_rel_att   * self._noisy_succ(0.9990 * afact["succ"]))
                pdu_mod_succ   = int(pdu_mod_att   * self._noisy_succ(0.9960 * afact["succ"]))

                # ══ 5.2.7  AUTHENTICATION / SECURITY (AUTH) ═══════════════
                auth_att     = init_reg_att + inter_mob_reg_att + intra_mob_reg_att
                nas_sec_att  = auth_att
                auth_succ    = int(auth_att * self._noisy_succ(0.9985 * afact["succ"]))
                auth_fail    = auth_att - auth_succ
                nas_sec_succ = int(nas_sec_att * self._noisy_succ(0.9990 * afact["succ"]))

                # Per-slice auth count (proportional)
                auth_cls = {cls: max(0, int(auth_att * _svc_mix[cls])) for cls in _svc_mix}

                # ══ 5.2.8  N1/N2 INTERFACE LOAD (N1N2) ════════════════════
                n1n2_total = 0.0
                for ev_key, ev_dict in EVENT_DATA.items():
                    for cls, n_ues_c in _ues_cls.items():
                        if n_ues_c <= 0:
                            continue
                        scale_c = SLICE_PROC_SCALE.get(cls, {}).get(ev_key, 1.0)
                        mean_ev = (ev_dict["bh_rate"] * scale_c * n_ues_c
                                   * STEP_MIN / 60.0) / _amf_n
                        mean_ev *= _load_cls.get(cls, load) * anom_req
                        n1n2_total += mean_ev * ev_dict["msgs"]

                n1_msgs_sent = int(StatUtils.sample_nb(n1n2_total * 0.50, 40, amf_rng))
                n1_msgs_recv = int(StatUtils.sample_nb(n1n2_total * 0.50 * amf_rng.uniform(0.95, 1.05), 40, amf_rng))
                n2_msgs_sent = int(StatUtils.sample_nb(n1n2_total * 0.50, 40, amf_rng))
                n2_msgs_recv = int(StatUtils.sample_nb(n1n2_total * 0.50 * amf_rng.uniform(0.95, 1.05), 40, amf_rng))
                ngap_active  = int(amf_rng.uniform(48, 56))

                # ══ 5.2.9  RESOURCE / PERFORMANCE (RES) ══════════════════

                # ── Aggregate M/M/c queueing model (all slices combined) ──
                total_events = (total_reg_att + total_ho_att + paging_att
                                + pdu_estab_att + n2_rel_att)
                lam_s    = max(total_events / self._SLOT_S, 1e-6)
                mu_s     = self._MU_SRV * vcpus * inst_cap
                _Ts, Wq, rho_mmc = StatUtils.mmc_sojourn(lam_s, mu_s, vcpus)

                # ── CPU-weight-adjusted rho (needed for M/M/1 latency coupling)
                # Compute rho_cpu early so it can drive the per-message NAS latency.
                # This mirrors the computation at the CPU model section below.
                def _cpu_w_slice_early(att_count: int, ev_key: str) -> float:
                    w = EVENT_DATA[ev_key]["cpu_w"]
                    return sum(
                        att_count * _svc_mix[cls] * w * SLICE_CPU_MULT[cls]
                        for cls in _svc_mix
                    )
                _cpu_equiv_early = (
                    _cpu_w_slice_early(init_reg_att,                          "Initial Registration")     +
                    _cpu_w_slice_early(mob_reg_att,                           "Inter-AMF Mobility Reg")   +
                    _cpu_w_slice_early(per_reg_att,                           "Periodic Registration")    +
                    _cpu_w_slice_early(srv_req_att,                           "Service Request")          +
                    _cpu_w_slice_early(n2_rel_att,                            "N2 Release")               +
                    _cpu_w_slice_early(intra_xn_ho_att,                       "Intra-AMF Xn Handover")   +
                    _cpu_w_slice_early(intra_n2_ho_att + inter_n2_ho_att,     "Inter-AMF N2 Handover")   +
                    _cpu_w_slice_early(pdu_estab_att,                         "PDU Session Establishment")+
                    _cpu_w_slice_early(auth_att,                              "Initial Registration")     +
                    _cpu_w_slice_early(paging_att,                            "PS Paging")
                )
                _rho_cpu_early = float(np.clip(
                    _cpu_equiv_early / max(_SLOT_S_eff * self._MU_SRV * vcpus * inst_cap, 1e-9),
                    0.01, 0.95
                ))

                # ── M/M/1 per-message NAS latency (paper eq. 21) ──────────────
                # Wq = rho / (mu*(1-rho)); 1/mu = 0.35ms → E[Wq] ≈ 0.9ms at rho=0.72.
                # rho_cpu_early (instruction-count based) correctly couples latency
                # to CPU load, producing r(CpuUtil, Latency_ms) ≈ +0.86 on normal rows.
                _MU_NAS_MS  = 1.0 / 0.35          # 2.857 messages/ms
                _Wq_mm1_ms  = (_rho_cpu_early * afact["lat"]
                                / (_MU_NAS_MS * max(1.0 - _rho_cpu_early, 0.01)))
                T_QUEUE_ms  = _Wq_mm1_ms

                # ── Jackson network per-slice throughput ──────────────────
                lam_per_cls = {
                    cls: max(
                        sum(self._slice_mean(ek, cls, _ues_cls[cls], _amf_n,
                                             _load_cls[cls])
                            for ek in EVENT_DATA) / self._SLOT_S, 1e-9)
                    for cls in _svc_mix
                }
                jackson_tput = StatUtils.jackson_throughput(
                    lam_per_cls, mu_s, vcpus
                )

                # ── Per-procedure NAS latency ─────────────────────────────
                def _proc_lat(base_ms: float) -> float:
                    total = T_QUEUE_ms + base_ms * afact["lat"]
                    return float(StatUtils.sample_lognormal_ms(total, 0.12, amf_rng))  # cv=0.12 per paper §IV-G

                lat_init_reg_ms  = _proc_lat(21.9)
                lat_mob_reg_ms   = _proc_lat(14.9)
                lat_per_reg_ms   = _proc_lat(4.9)
                lat_srv_req_ms   = _proc_lat(6.9)
                lat_n2_rel_ms    = _proc_lat(1.9)
                lat_xn_ho_ms     = _proc_lat(8.9)
                lat_n2_ho_ms     = _proc_lat(16.9)
                lat_pdu_estab_ms = _proc_lat(18.9)
                lat_auth_ms      = _proc_lat(11.9)
                lat_paging_ms    = _proc_lat(2.9)

                _lat_counts = [
                    (init_reg_att,                   lat_init_reg_ms),
                    (mob_reg_att,                    lat_mob_reg_ms),
                    (per_reg_att,                    lat_per_reg_ms),
                    (srv_req_att,                    lat_srv_req_ms),
                    (n2_rel_att,                     lat_n2_rel_ms),
                    (intra_xn_ho_att,                lat_xn_ho_ms),
                    (intra_n2_ho_att+inter_n2_ho_att,lat_n2_ho_ms),
                    (pdu_estab_att,                  lat_pdu_estab_ms),
                    (auth_att,                       lat_auth_ms),
                    (paging_att,                     lat_paging_ms),
                ]
                _total_cnt = sum(c for c, _ in _lat_counts)
                lat_ms = (sum(c * l for c, l in _lat_counts) / _total_cnt
                          if _total_cnt > 0 else T_QUEUE_ms + 10.0)

                # ── Aggregate CPU model (procedure-weighted + M/M/c blend) ─
                # Weighted by per-slice CPU multiplier (SLICE_CPU_MULT) so that
                # changing eMBB/mMTC/URLLC proportions is correctly reflected.
                # Each event count is split proportionally by _svc_mix, then
                # multiplied by the slice CPU multiplier before summing.
                def _cpu_w_slice(att_count: int, ev_key: str) -> float:
                    w = EVENT_DATA[ev_key]["cpu_w"]
                    return sum(
                        att_count * _svc_mix[cls] * w * SLICE_CPU_MULT[cls]
                        for cls in _svc_mix
                    )

                cpu_equiv = (
                    _cpu_w_slice(init_reg_att,                          "Initial Registration")     +
                    _cpu_w_slice(mob_reg_att,                           "Inter-AMF Mobility Reg")   +
                    _cpu_w_slice(per_reg_att,                           "Periodic Registration")    +
                    _cpu_w_slice(srv_req_att,                           "Service Request")          +
                    _cpu_w_slice(n2_rel_att,                            "N2 Release")               +
                    _cpu_w_slice(intra_xn_ho_att,                       "Intra-AMF Xn Handover")   +
                    _cpu_w_slice(intra_n2_ho_att + inter_n2_ho_att,     "Inter-AMF N2 Handover")   +
                    _cpu_w_slice(pdu_estab_att,                         "PDU Session Establishment")+
                    _cpu_w_slice(auth_att,                              "Initial Registration")     +
                    _cpu_w_slice(paging_att,                            "PS Paging")
                )
                # Rate-based CPU rho — slot-size independent
                lam_cpu_eff   = cpu_equiv / _SLOT_S_eff           # events/s
                vcpu_cap_slot = self._MU_SRV * vcpus * inst_cap   # events/s capacity
                rho_cpu       = lam_cpu_eff / max(vcpu_cap_slot, 1e-9)
                rho_blend     = 0.65 * rho_cpu + 0.35 * rho_mmc
                cpu_base      = min(100.0, rho_blend * 100.0 * afact["cpu"])
                cpu_util      = float(np.clip(cpu_base + amf_rng.normal(0.0, 1.5), 0.5, 99.5))

                # ── Aggregate memory model ────────────────────────────────
                active_pdu_sess = connected * 2.5
                queue_len_raw   = max(0.0, lam_s * Wq * self._SLOT_S)
                mem_base_mb     = 512.0
                mem_ctx_mb      = 0.005 * active_ctx
                mem_pdu_mb      = 0.001 * active_pdu_sess
                mem_sigbuf_mb   = 0.04  * queue_len_raw
                mem_auth_mb     = min(180.0, 0.08 * auth_att)
                mem_total_raw   = mem_base_mb + mem_ctx_mb + mem_pdu_mb + mem_sigbuf_mb + mem_auth_mb
                mem_total_mb    = mem_total_raw * afact["mem"]
                mem_util        = float(np.clip(
                    100.0 * mem_total_mb / self.mem_max_mb + amf_rng.normal(0.0, 0.6), 1.0, 99.5
                ))

                # ── NEW: Per-slice CPU, Memory, Latency ──────────────
                # Build per-slice event-count dicts for CPU computation
                slice_cpu_pct: Dict[str, float] = {}
                slice_mem_mb:  Dict[str, float] = {}
                slice_lat_ms:  Dict[str, float] = {}

                for cls in _svc_mix:
                    # Per-slice event counts (proportional slice_mean weighting)
                    ev_counts_cls: Dict[str, int] = {}
                    for ev_key in EVENT_DATA:
                        lam_cls_ev = self._slice_mean(ev_key, cls, _ues_cls[cls], _amf_n, _load_cls[cls])
                        ev_counts_cls[ev_key] = max(0, int(StatUtils.sample_nb(
                            lam_cls_ev * anom_req, EVENT_DATA[ev_key]["nb_k"], amf_rng
                        )))
                    slice_cpu_pct[cls] = self._compute_slice_cpu(
                        ev_counts_cls, vcpus, inst_cap, afact, cls
                    )
                    slice_mem_mb[cls]  = self._compute_slice_mem(
                        ues_conn_cls[cls], auth_cls[cls], queue_len_raw, afact, cls
                    )
                    slice_lat_ms[cls]  = self._compute_slice_lat(Wq, afact, cls, amf_rng)

                # CPU cycles per message (efficiency metric)
                total_msgs        = max(n1_msgs_sent + n2_msgs_sent, 1)
                cpu_cycles_per_msg = (cpu_equiv * 1e6) / total_msgs   # approx instruction equiv

                # Memory bytes per UE
                mem_bytes_per_ue = (mem_total_mb * 1024 * 1024) / max(active_ctx, 1)

                # Slice load entropy
                slice_entropy = self._slice_entropy(_load_cls)

                throughput_mps = (n1_msgs_sent + n2_msgs_sent) / self._SLOT_S
                queue_len      = int(max(0, amf_rng.poisson(max(queue_len_raw, 1e-6))))
                active_workers = int(np.clip(rho_mmc * vcpus * 10, 0, vcpus * 10))

                # ─── Assemble row (3GPP TS 28.552 naming conventions) ────
                row = {
                    # ── Metadata ────────────────────────────────────────
                    "timestamp":          ts,
                    "amf_instance_id":    inst_id,
                    "is_anomaly":         int(is_anom),
                    "anomaly_type":       anom_type,
                    "anomaly_intensity":  sc.get("intensity", "none") if is_anom else "none",
                    "anomaly_sigmoid_w":  round(sig_w, 4),
                    "composite_load":     round(float(load), 4),
                    "rho":                round(float(rho_mmc), 4),

                    # ── 5.2.1 Registration Management ──────────────────
                    "RM.RegReqAtt":          total_reg_att,
                    "RM.RegReqSucc":         total_reg_succ,
                    "RM.RegReqFail":         total_reg_att - total_reg_succ,
                    "RM.RegSuccRate":        round(100.0 * total_reg_succ / max(total_reg_att,1), 3),
                    "RM.InitRegReqAtt":      init_reg_att,
                    "RM.InitRegReqSucc":     init_reg_succ,
                    "RM.MobilityRegReqAtt":  mob_reg_att,
                    "RM.MobilityRegReqSucc": mob_reg_succ,
                    "RM.PeriodicRegReqAtt":  per_reg_att,
                    "RM.PeriodicRegReqSucc": per_reg_succ,
                    "RM.DeregReqAtt":        dereg_att,
                    "RM.DeregReqSucc":       dereg_succ,

                    # ── 5.2.2 Connection Management ────────────────────
                    "CM.ServiceReqAtt":      srv_req_att,
                    "CM.ServiceReqSucc":     srv_req_succ,
                    "CM.ServiceReqSuccRate": round(100.0 * srv_req_succ / max(srv_req_att,1), 3),
                    "CM.N2RelAtt":           n2_rel_att,
                    "CM.N2RelSucc":          n2_rel_succ,

                    # ── 5.2.3 Mobility Management ──────────────────────
                    "MM.HoReqAtt":           total_ho_att,
                    "MM.HoReqSucc":          total_ho_succ,
                    "MM.HoSuccRate":         round(100.0 * total_ho_succ / max(total_ho_att,1), 3),
                    "MM.XnHoReqAtt":         intra_xn_ho_att,
                    "MM.XnHoReqSucc":        intra_xn_ho_succ,
                    "MM.N2IntraHoReqAtt":    intra_n2_ho_att,
                    "MM.N2IntraHoReqSucc":   intra_n2_ho_succ,
                    "MM.N2InterHoReqAtt":    inter_n2_ho_att,
                    "MM.N2InterHoReqSucc":   inter_n2_ho_succ,
                    "MM.EPS2fiveGSMobAtt":   eps5g_mob,
                    "MM.fiveGS2EPSMobAtt":   five_eps_mob,
                    "MM.N26HoAtt":           inter_n26_ho,

                    # ── 5.2.4 Paging ───────────────────────────────────
                    "PAG.PagingReqAtt":      paging_att,
                    "PAG.PagingReqSucc":     paging_succ,
                    "PAG.PagingSuccRate":    round(100.0 * paging_succ / max(paging_att,1), 3),
                    "PAG.PagingDiscarded":   paging_disc,
                    "PAG.PagingRetry":       paging_retry,
                    "PAG.UeInIdleMode":      int(idle),

                    # ── 5.2.5 UE Context ────────────────────────────────
                    "UC.UeContextCreated":   ctx_created,
                    "UC.UeContextReleased":  ctx_released,
                    "UC.UeContextModified":  ctx_modified,
                    "UC.ActiveUeContext":    active_ctx,
                    "UC.MaxUeContextCap":    max_ctx_cap,
                    "UC.ContextUtilRate":    round(min(100.0, 100.0 * active_ctx / max(max_ctx_cap,1)), 3),

                    # ── 5.2.6 PDU Session ───────────────────────────────
                    "SM.PduSessEstabAtt":       pdu_estab_att,
                    "SM.PduSessEstabSucc":      pdu_estab_succ,
                    "SM.PduSessEstabSuccRate":  round(100.0 * pdu_estab_succ / max(pdu_estab_att,1), 3),
                    "SM.PduSessRelAtt":         pdu_rel_att,
                    "SM.PduSessRelSucc":        pdu_rel_succ,
                    "SM.PduSessModAtt":         pdu_mod_att,
                    "SM.PduSessModSucc":        pdu_mod_succ,
                    "SM.VoNRAtt":               vonr_att,
                    "SM.EPSFallbackAtt":        eps_fb_att,
                    "SM.SMSAtt":                sms_att,

                    # ── 5.2.7 Authentication ────────────────────────────
                    "AUTH.AuthProcAtt":         auth_att,
                    "AUTH.AuthProcSucc":        auth_succ,
                    "AUTH.AuthProcFail":        auth_fail,
                    "AUTH.AuthSuccRate":        round(100.0 * auth_succ / max(auth_att,1), 3),
                    "AUTH.NasSecModeAtt":       nas_sec_att,
                    "AUTH.NasSecModeSucc":      nas_sec_succ,

                    # ── 5.2.8 N1/N2 Interface Load ───────────────────────
                    "N1N2.N1MsgSent":           n1_msgs_sent,
                    "N1N2.N1MsgRecv":           n1_msgs_recv,
                    "N1N2.N2MsgSent":           n2_msgs_sent,
                    "N1N2.N2MsgRecv":           n2_msgs_recv,
                    "N1N2.NgapConnActive":       ngap_active,
                    "N1N2.TotalMsgLoad":         int(n1n2_total),

                    # ── 5.2.9 Resource / Performance (aggregate) ─────────
                    "RES.CpuUtil":              round(cpu_util, 2),
                    "RES.RhoCPU_weighted":      round(float(min(rho_blend, 1.5)), 4),
                    "RES.MemUtil":              round(mem_util, 2),
                    "RES.MemTotal_MB":          round(mem_total_mb, 1),
                    "RES.MemBase_MB":           round(mem_base_mb, 1),
                    "RES.MemCtx_MB":            round(mem_ctx_mb, 1),
                    "RES.MemPDU_MB":            round(mem_pdu_mb, 1),
                    "RES.MemSigBuf_MB":         round(mem_sigbuf_mb, 2),
                    "RES.MemAuthCache_MB":      round(mem_auth_mb, 1),
                    "RES.Latency_ms":           round(lat_ms, 3),
                    "RES.Wq_ms":                round(T_QUEUE_ms, 3),
                    "RES.Lat_InitReg_ms":       round(lat_init_reg_ms, 3),
                    "RES.Lat_MobReg_ms":        round(lat_mob_reg_ms, 3),
                    "RES.Lat_PerReg_ms":        round(lat_per_reg_ms, 3),
                    "RES.Lat_SrvReq_ms":        round(lat_srv_req_ms, 3),
                    "RES.Lat_N2Rel_ms":         round(lat_n2_rel_ms, 3),
                    "RES.Lat_XnHO_ms":          round(lat_xn_ho_ms, 3),
                    "RES.Lat_N2HO_ms":          round(lat_n2_ho_ms, 3),
                    "RES.Lat_PduEstab_ms":      round(lat_pdu_estab_ms, 3),
                    "RES.Lat_Auth_ms":          round(lat_auth_ms, 3),
                    "RES.Lat_Paging_ms":        round(lat_paging_ms, 3),
                    "RES.Throughput_mps":       round(throughput_mps, 3),
                    "RES.QueueLength":          queue_len,
                    "RES.ActiveWorkers":        active_workers,
                    "RES.ActiveUEs":            active_ues,
                    "RES.ConnectedUEs":         connected,
                    "RES.IdleUEs":              int(idle),

                    # ── NEW: Per-Slice Resource KPIs ────────────────
                    "RES.CPU_eMBB_pct":         round(slice_cpu_pct["eMBB"],  2),
                    "RES.CPU_mMTC_pct":         round(slice_cpu_pct["mMTC"],  2),
                    "RES.CPU_URLLC_pct":        round(slice_cpu_pct["URLLC"], 2),
                    "RES.Mem_eMBB_MB":          round(slice_mem_mb["eMBB"],   2),
                    "RES.Mem_mMTC_MB":          round(slice_mem_mb["mMTC"],   2),
                    "RES.Mem_URLLC_MB":         round(slice_mem_mb["URLLC"],  2),
                    "RES.Lat_eMBB_ms":          round(slice_lat_ms["eMBB"],   3),
                    "RES.Lat_mMTC_ms":          round(slice_lat_ms["mMTC"],   3),
                    "RES.Lat_URLLC_ms":         round(slice_lat_ms["URLLC"],  3),
                    "RES.CpuCyclesPerMsg":       round(cpu_cycles_per_msg, 1),
                    "RES.MemBytesPerUE":         round(mem_bytes_per_ue, 1),
                    "RES.SliceLoadEntropy":      round(slice_entropy, 4),
                    # Jackson per-class throughput
                    "RES.Jackson_eMBB_eps":      round(jackson_tput["eMBB"],  3),
                    "RES.Jackson_mMTC_eps":      round(jackson_tput["mMTC"],  3),
                    "RES.Jackson_URLLC_eps":     round(jackson_tput["URLLC"], 3),
                    # SLA breach flags (bool int)
                    "SLA.eMBB_breach":           int(slice_lat_ms["eMBB"]  > SLICE_LAT_PARAMS["eMBB"]["sla_ms"]),
                    "SLA.mMTC_breach":           int(slice_lat_ms["mMTC"]  > SLICE_LAT_PARAMS["mMTC"]["sla_ms"]),
                    "SLA.URLLC_breach":          int(slice_lat_ms["URLLC"] > SLICE_LAT_PARAMS["URLLC"]["sla_ms"]),
                }
                all_rows.append(row)

            # Report progress after each AMF instance
            if progress_callback is not None:
                progress_callback((amf_idx + 1) / _amf_n * 100.0)

        df = pd.DataFrame(all_rows)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        return df

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 7 — Dataset Validation
# ──────────────────────────────────────────────────────────────────────

In [8]:
# validate_dataset + export_dataset
def validate_dataset(df: pd.DataFrame, amf_instances: int = 1) -> dict:
    """
    Structural checks run against the *internal* (116-column) DataFrame
    immediately after generation.

    Returns a dict mapping check name → bool.  All must be True before export.

    Checks
    ------
    row_count_correct   : will be set by the caller after computing expected rows
    no_nan_in_key_cols  : core label + resource columns contain no NaN
    rates_in_range      : success-rate columns are in [0, 100]
    counts_non_negative : all count columns are >= 0
    anomaly_labels_ok   : is_anomaly is 0/1; anomaly_type is non-null string
    correct_amf_instances : number of unique AMF instance IDs matches amf_instances
    timestamps_ordered  : timestamp column is strictly monotonically increasing
    cpu_in_range        : RES.CpuUtil values are in [0, 100]
    mem_in_range        : RES.MemUtil values are in [0, 100]
    succ_le_att         : RM.RegReqSucc <= RM.RegReqAtt for every row
    """
    results = {}

    # ── row_count_correct ────────────────────────────────────────────────────
    # Placeholder — caller sets this after computing (duration * 60 // step) * instances
    results["row_count_correct"] = True

    # ── no_nan_in_key_cols ───────────────────────────────────────────────────
    key_cols = [c for c in [
        "timestamp", "amf_instance_id", "is_anomaly", "anomaly_type",
        "RES.CpuUtil", "RES.MemUtil", "RES.Latency_ms",
        "RM.RegReqAtt", "RM.RegReqSucc",
    ] if c in df.columns]
    results["no_nan_in_key_cols"] = bool(df[key_cols].notna().all().all())

    # ── rates_in_range ───────────────────────────────────────────────────────
    rate_cols = [c for c in df.columns if "Rate" in c or "Util" in c]
    if rate_cols:
        results["rates_in_range"] = bool(
            (df[rate_cols] >= 0).all().all() and (df[rate_cols] <= 100).all().all()
        )
    else:
        results["rates_in_range"] = True

    # ── counts_non_negative ──────────────────────────────────────────────────
    count_cols = [c for c in df.columns
                  if any(t in c for t in ["Att", "Succ", "Fail", "Req", "Msg"])]
    if count_cols:
        results["counts_non_negative"] = bool((df[count_cols] >= 0).all().all())
    else:
        results["counts_non_negative"] = True

    # ── anomaly_labels_ok ────────────────────────────────────────────────────
    results["anomaly_labels_ok"] = bool(
        df["is_anomaly"].isin([0, 1]).all()
        and df["anomaly_type"].notna().all()
    )

    # ── correct_amf_instances ────────────────────────────────────────────────
    results["correct_amf_instances"] = (
        df["amf_instance_id"].nunique() == amf_instances
    )

    # ── timestamps_ordered ───────────────────────────────────────────────────
    ts = pd.to_datetime(df["timestamp"])
    results["timestamps_ordered"] = bool(ts.is_monotonic_increasing)

    # ── cpu_in_range ─────────────────────────────────────────────────────────
    if "RES.CpuUtil" in df.columns:
        results["cpu_in_range"] = bool(
            (df["RES.CpuUtil"] >= 0).all() and (df["RES.CpuUtil"] <= 100).all()
        )
    else:
        results["cpu_in_range"] = True

    # ── mem_in_range ─────────────────────────────────────────────────────────
    if "RES.MemUtil" in df.columns:
        results["mem_in_range"] = bool(
            (df["RES.MemUtil"] >= 0).all() and (df["RES.MemUtil"] <= 100).all()
        )
    else:
        results["mem_in_range"] = True

    # ── succ_le_att ──────────────────────────────────────────────────────────
    if "RM.RegReqSucc" in df.columns and "RM.RegReqAtt" in df.columns:
        results["succ_le_att"] = bool(
            (df["RM.RegReqSucc"] <= df["RM.RegReqAtt"]).all()
        )
    else:
        results["succ_le_att"] = True

    return results


print("✓ validate_dataset() defined.")


def export_dataset(df: pd.DataFrame, base_path: str) -> Tuple[str, str]:
    # ── Strip internal generator metadata columns before export ──────────
    # These columns are used internally by the generator/anomaly engine and
    # must NOT appear in the public dataset — they would cause data leakage
    # in any ML model trained on the dataset:
    #   anomaly_sigmoid_w : = 0.0 for ALL normal rows, > 0 for ALL anomaly rows
    #                         → perfect label proxy, not a real AMF KPI
    #   anomaly_intensity : string label "none"/"moderate"/etc.
    #                         → direct anomaly metadata, not a KPI
    #   composite_load    : internal load scalar driving the anomaly engine
    #   rho               : raw M/M/c utilisation before noise — internal calc
    _INTERNAL_COLS = {
        "anomaly_sigmoid_w",
        "anomaly_intensity",
        "composite_load",
        "rho",
    }
    export_cols = [c for c in df.columns if c not in _INTERNAL_COLS]
    df_export   = df[export_cols]

    csv_p  = base_path + ".csv"
    json_p = base_path + ".json"
    df_export.to_csv(csv_p, index=False)
    df_export.to_json(json_p, orient="records", date_format="iso", indent=2)
    print(f"  CSV  → {csv_p}  ({len(export_cols)} columns, "
          f"{len(_INTERNAL_COLS & set(df.columns))} internal cols stripped)")
    print(f"  JSON → {json_p}")
    return csv_p, json_p


def write_metadata(df: pd.DataFrame, path: str, config: Dict) -> None:
    meta = {
        "generator":       "amf_synthetic_dataset",
        "3gpp_standard":   "TS 28.552 v19.6.0",
        "generated_at":    datetime.utcnow().isoformat(),
        "config":          {k: str(v) for k, v in config.items()},
        "duration_hours":  DURATION_HOURS,
        "step_min":        self.step_min if hasattr(self, "step_min") else STEP_MIN,
        "total_rows":      len(df),
        "columns":         list(df.columns),
        "anomaly_rows":    int(df["is_anomaly"].sum()),
        "sla_breach_eMBB": int(df["SLA.eMBB_breach"].sum()),
        "sla_breach_mMTC": int(df["SLA.mMTC_breach"].sum()),
        "sla_breach_URLLC":int(df["SLA.URLLC_breach"].sum()),
        "stats": {c: {"mean": round(float(df[c].mean()),4),
                       "std":  round(float(df[c].std()),4),
                       "p95":  round(float(df[c].quantile(0.95)),4)}
                  for c in ["RES.CpuUtil","RES.MemUtil","RES.Latency_ms",
                              "RES.Lat_eMBB_ms","RES.Lat_mMTC_ms","RES.Lat_URLLC_ms"]
                  if c in df.columns},
    }
    with open(path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  Metadata → {path}")

✓ validate_dataset() defined.


In [9]:
# Leakage-prevention helper
PUBLIC_DROP_COLUMNS = ['anomaly_sigmoid_w', 'anomaly_intensity', 'composite_load', 'rho']

def build_public_release(df):
    drop_cols = [c for c in PUBLIC_DROP_COLUMNS if c in df.columns]
    return df.drop(columns=drop_cols).copy(), drop_cols



## Part 2 — Generate the 60-day SCOPE-5G Dataset (~30–60 s)

In [10]:
# Generate the released configuration: N_AMF=1, 60 days, 100k UEs, seed=42, with anomalies
import time
t0 = time.time()
gen = AMFDatasetGenerator(
    seed=42,
    duration_hours=60*24,
    step_min=15,
    ue_embb=70_000,
    ue_mmtc=20_000,
    ue_urllc=10_000,
    amf_instances=1,
    include_anomalies=True,
    anomaly_scenarios=ANOMALY_SCENARIOS,
)
df_internal = gen.generate()
df, _ = build_public_release(df_internal)
print(f'Generation done in {time.time()-t0:.1f}s; df={df.shape}, anomalies={df["is_anomaly"].sum()}')

Generation done in 57.9s; df=(5760, 112), anomalies=162


## Part 3 — LSTM-Autoencoder training and evaluation

In [11]:
# Imports for the LSTM-AE experiment
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

tf.random.set_seed(42)
np.random.seed(42)

# Identify the 102 TS 28.552 native columns
TS28552_PREFIXES = ('RM.', 'CM.', 'MM.', 'PAG.', 'UC.', 'SM.', 'AUTH.', 'N1N2.', 'RES.')
EXCLUDE_DERIVED = {
    'RES.Jackson_eMBB_eps', 'RES.Jackson_mMTC_eps', 'RES.Jackson_URLLC_eps',
    'SLA.eMBB_breach', 'SLA.mMTC_breach', 'SLA.URLLC_breach',
}
FEATURE_COLS = [c for c in df.columns
                if c.startswith(TS28552_PREFIXES) and c not in EXCLUDE_DERIVED]
print(f'{len(FEATURE_COLS)} TS 28.552 native columns (expected 102)')
assert len(FEATURE_COLS) == 102, f'Expected 102, got {len(FEATURE_COLS)}'

# Day index (1-based from dataset start)
df = df.sort_values('timestamp').reset_index(drop=True)
df['day_1based'] = (pd.to_datetime(df['timestamp']).dt.dayofyear -
                    pd.to_datetime(df['timestamp']).dt.dayofyear.min() + 1).astype(int)

102 TS 28.552 native columns (expected 102)


In [12]:
# Split helpers
WINDOW = 96  # 24h at 15-min granularity

def split_masks(df, mode='S1'):
    day = df['day_1based']
    if mode == 'S1':
        train_mask = day.isin(list(range(1, 3)) + list(range(50, 61)))
        test_mask  = day.isin(list(range(3, 50)))
    elif mode == 'S2':
        train_mask = day.isin([1, 2])
        test_mask  = day.isin(list(range(3, 61)))
    else:
        raise ValueError(mode)
    return train_mask.values, test_mask.values

def build_lstm_ae(seq_len, n_features, latent=64, units=128):
    inp = layers.Input(shape=(seq_len, n_features))
    x = layers.LSTM(units, return_sequences=True)(inp)
    x = layers.LSTM(units, return_sequences=False)(x)
    z = layers.Dense(latent, activation='relu')(x)
    x = layers.RepeatVector(seq_len)(z)
    x = layers.LSTM(units, return_sequences=True)(x)
    x = layers.LSTM(units, return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(n_features))(x)
    model = Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

In [13]:
def evaluate_lstm_ae(mode='S1', epochs=20, batch=64):
    train_mask, test_mask = split_masks(df, mode)
    X = df[FEATURE_COLS].values.astype(np.float32)
    y = df['is_anomaly'].values.astype(int)

    scaler = StandardScaler().fit(X[train_mask])
    Xs = scaler.transform(X)

    # Build training windows: only from clean rows in train_mask
    train_idx = np.where(train_mask & (y == 0))[0]
    train_windows = []
    for i in train_idx:
        if i - WINDOW + 1 >= 0:
            w = Xs[i-WINDOW+1 : i+1]
            if not np.isnan(w).any():
                train_windows.append(w)
    train_X = np.stack(train_windows).astype(np.float32)
    print(f'[{mode}] Training windows: {train_X.shape}')

    model = build_lstm_ae(WINDOW, len(FEATURE_COLS))
    model.fit(train_X, train_X, epochs=epochs, batch_size=batch,
              validation_split=0.1, verbose=0)

    # Score every test row using the window that ends at that row
    test_idx = np.where(test_mask)[0]
    valid_test_idx = [i for i in test_idx if i - WINDOW + 1 >= 0]
    test_windows = np.stack([Xs[i-WINDOW+1 : i+1] for i in valid_test_idx]).astype(np.float32)
    recon = model.predict(test_windows, batch_size=batch, verbose=0)
    # Score = per-window mean squared reconstruction error on the LAST slot
    scores = np.mean((test_windows[:, -1, :] - recon[:, -1, :])**2, axis=1)

    # Threshold: 99th percentile of clean training reconstruction errors
    sample = train_X[:min(500, len(train_X))]
    train_recon = model.predict(sample, batch_size=batch, verbose=0)
    train_scores = np.mean((sample[:, -1, :] - train_recon[:, -1, :])**2, axis=1)
    thr = np.percentile(train_scores, 99)

    yt = y[valid_test_idx]
    yh = (scores >= thr).astype(int)
    return {
        'split':   mode,
        'Prec':    precision_score(yt, yh, zero_division=0),
        'Rec':     recall_score(yt, yh, zero_division=0),
        'F1':      f1_score(yt, yh, zero_division=0),
        'AUC':     roc_auc_score(yt, scores),
        'n_test':  len(yt),
        'n_anom':  int(yt.sum()),
    }

In [14]:
# Also run Isolation Forest on the S2 split (the IF S2 row of Table 10)
def evaluate_if_s2():
    train_mask, test_mask = split_masks(df, 'S2')
    X = df[FEATURE_COLS].values.astype(np.float32)
    y = df['is_anomaly'].values.astype(int)
    scaler = StandardScaler().fit(X[train_mask])
    Xs = scaler.transform(X)
    iforest = IsolationForest(n_estimators=200, contamination=0.028, random_state=42)
    iforest.fit(Xs[train_mask & (y == 0)])
    yhat = (iforest.predict(Xs[test_mask]) == -1).astype(int)
    s = -iforest.score_samples(Xs[test_mask])
    yt = y[test_mask]
    return {
        'split':  'S2',
        'Prec':   precision_score(yt, yhat, zero_division=0),
        'Rec':    recall_score(yt, yhat, zero_division=0),
        'F1':     f1_score(yt, yhat, zero_division=0),
        'AUC':    roc_auc_score(yt, s),
    }

In [15]:
# Run all three measurements
print('=== Training LSTM-AE (S1 default split) — this is the slow step ===')
res_lstm_s1 = evaluate_lstm_ae('S1', epochs=20)
print('S1 LSTM-AE:', res_lstm_s1)

print()
print('=== Training LSTM-AE (S2 temporal-gen split) ===')
res_lstm_s2 = evaluate_lstm_ae('S2', epochs=20)
print('S2 LSTM-AE:', res_lstm_s2)

print()
print('=== Isolation Forest on S2 (no training needed beyond fit) ===')
res_if_s2 = evaluate_if_s2()
print('S2 IF:', res_if_s2)

print()
print('=' * 70)
print('NUMBERS TO PASTE INTO TABLE 7:')
print('=' * 70)
print(f'[Method-comparison block]   LSTM-AE (96-slot, 64-dim)  | '
      f'Prec={res_lstm_s1["Prec"]:.3f}  Rec={res_lstm_s1["Rec"]:.3f}  '
      f'F1={res_lstm_s1["F1"]:.3f}  AUC={res_lstm_s1["AUC"]:.3f}')
print(f'[Temporal-gen.  block]      IF (200 trees, S2)         | '
      f'Prec={res_if_s2["Prec"]:.3f}  Rec={res_if_s2["Rec"]:.3f}  '
      f'F1={res_if_s2["F1"]:.3f}  AUC={res_if_s2["AUC"]:.3f}')
print(f'[Temporal-gen.  block]      LSTM-AE (96-slot, 64-dim)  | '
      f'Prec={res_lstm_s2["Prec"]:.3f}  Rec={res_lstm_s2["Rec"]:.3f}  '
      f'F1={res_lstm_s2["F1"]:.3f}  AUC={res_lstm_s2["AUC"]:.3f}')

=== Training LSTM-AE (S1 default split) — this is the slow step ===
[S1] Training windows: (1153, 96, 102)
S1 LSTM-AE: {'split': 'S1', 'Prec': 0.7287234042553191, 'Rec': 0.845679012345679, 'F1': 0.7828571428571428, 'AUC': np.float64(0.9362792677735207), 'n_test': 4512, 'n_anom': 162}

=== Training LSTM-AE (S2 temporal-gen split) ===
[S2] Training windows: (97, 96, 102)
S2 LSTM-AE: {'split': 'S2', 'Prec': 0.7567567567567568, 'Rec': 0.8641975308641975, 'F1': 0.8069164265129684, 'AUC': np.float64(0.9532595241683909), 'n_test': 5568, 'n_anom': 162}

=== Isolation Forest on S2 (no training needed beyond fit) ===
S2 IF: {'split': 'S2', 'Prec': 0.49645390070921985, 'Rec': 0.8641975308641975, 'F1': 0.6306306306306306, 'AUC': np.float64(0.942542122835624)}

NUMBERS TO PASTE INTO TABLE 7:
[Method-comparison block]   LSTM-AE (96-slot, 64-dim)  | Prec=0.729  Rec=0.846  F1=0.783  AUC=0.936
[Temporal-gen.  block]      IF (200 trees, S2)         | Prec=0.496  Rec=0.864  F1=0.631  AUC=0.943
[Temporal-

## Numbers to insert in the manuscript

Copy the three lines above into the corresponding `[P]` cells of **Table 10** in `SCOPE-5G_clean_REVISED.tex`:

| Table 10 row | Block | What to paste |
|---|---|---|
| LSTM-AE (96-slot, 64-dim) | Method comparison (S1) | Prec / Rec / F1 / AUC from `res_lstm_s1` |
| IF (200 trees, cont=0.028, rs=42) | Temporal-generalization (S2) | Prec / Rec / F1 / AUC from `res_if_s2` |
| LSTM-AE (96-slot, 64-dim) | Temporal-generalization (S2) | Prec / Rec / F1 / AUC from `res_lstm_s2` |

All values are deterministic under `seed=42` modulo TensorFlow GPU non-determinism. If you re-run on the same Colab runtime, results are stable within ±0.005.